# LLM预训练、SFT和DPO的实现笔记

可使用Google Colab上使用T4 GPU进行训练

In [4]:
# 通用库
import torch
import torch.nn as nn
import evaluate
import numpy as np
import tqdm
import sys
from typing import Dict, Optional, Any, Dict, List, Optional, Union
from dataclasses import dataclass, field

# Huggingface Transformers系列库
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer, TrainerCallback
from transformers import AutoTokenizer, AutoConfig, DataCollatorForLanguageModeling
from transformers import GPT2Config, GPT2ForSequenceClassification, AutoModelForSequenceClassification
from transformers import PreTrainedTokenizerBase
from transformers import Adafactor, pipeline
from transformers import BitsAndBytesConfig
from transformers.utils import PaddingStrategy

from accelerate import Accelerator

from datasets import load_dataset, load_from_disk, concatenate_datasets, Dataset, DatasetDict

from peft import PeftModel, PeftConfig, LoraConfig
from peft import TaskType, get_peft_model, get_peft_config

from trl import SFTTrainer, DPOTrainer
from trl import AutoModelForCausalLMWithValueHead, PPOConfig, PPOTrainer, set_seed
from trl.core import LengthSampler
from trl.trainer import ConstantLengthDataset

Exception ignored in: <function _xla_gc_callback at 0x786ef42cc550>
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/jax/_src/lib/__init__.py", line 97, in _xla_gc_callback
    def _xla_gc_callback(*args):
KeyboardInterrupt: 


In [25]:
batch_size = 8
max_length = 256
max_steps = 1000
device = 'cuda:0'
# device = 'cuda'
lora_r = 8
debug_mode = False
use_pretrained_text_data = True
block_size = 256

In [26]:
import os
temp_path = './'
if os.path.exists('/content/drive/MyDrive'):
    # 如果路径不存在，则创建文件夹
    temp_path = '/content/drive/MyDrive/tinyllama-medical/'
    if not os.path.exists('/content/drive/MyDrive/tinyllama-medical'): #训练过程中所存放的网盘模型路径
      os.makedirs(temp_path)
      print("文件夹已创建")
else:
    print("使用本地路径")

In [27]:
# 模型名称
datasets_name = 'shibing624/medical'
model_pretrained_name = temp_path + 'tinyllama-medical-pretrained'
model_sft_name = temp_path + 'tinyllama-medical-SFT'
model_rm_name = temp_path + 'tinyllama-medical-RM'
model_ppo_name = temp_path + 'tinyllama-medical-PPO'
model_dpo_name = temp_path + 'tinyllama-medical-DPO'

# LLaMA 1B 使用原生LLaMA tokenizer对中文支持不友好，会添加很多额外的Token
model_base_name = 'PY007/TinyLlama-1.1B-Chat-v0.3'
tokenizer_name = 'PY007/TinyLlama-1.1B-Chat-v0.3'
model_name = 'PY007/TinyLlama-1.1B-Chat-v0.3'

if debug_mode:
    model_name = temp_path + './LLaMA_base_baby'

dataset_dir = './med_qa_textbook'  
data_cache_dir = 'temp_data_cache_dir'

In [28]:
# QLoRA
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [29]:

lora_full = ['embed_tokens', 'lm_head', 'q_proj', 'k_proj', 'v_proj', 'o_proj',
             'gate_proj','up_proj','down_proj']
lora_pretrained = ['embed_tokens', 'lm_head', 'q_proj', 'k_proj', 'v_proj', 'o_proj', 'down_proj']
lora_finetune = [ 'q_proj', 'k_proj', 'v_proj', 'o_proj', 'down_proj']

pretrained_lora_config = LoraConfig(
    task_type="CAUSAL_LM",  # 任务类型
    r=8,
    target_modules = lora_pretrained,  # 选择要量化的模块
    modules_to_save = None,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
)

lm_lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=8,
    target_modules = lora_finetune,
    modules_to_save = None,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
)

rm_lora_config = LoraConfig(
    task_type="SEQ_CLS",
    r=8,
    target_modules = lora_finetune,
    modules_to_save = None,
    lora_alpha=32,
    lora_dropout=0.05,
    inference_mode=False,  # inference_mode=True时，只有target_modules中的模块会被量化, 否则所有模块都会被量化
    bias="none",
)


# 中文tokenizer

In [30]:
# 加载tokenizer
IGNORE_INDEX = -100
DEFAULT_PAD_TOKEN = "[PAD]"
DEFAULT_EOS_TOKEN = "</s>"
DEFAULT_BOS_TOKEN = "<s>"
DEFAULT_UNK_TOKEN = "<unk>"

print(tokenizer_name)
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name, use_fast = False)
tokenizer.pad_token = tokenizer.eos_token # 原始LLaMA tokenizer 没有Pad Token， 统一用eos替换
print(tokenizer)


PY007/TinyLlama-1.1B-Chat-v0.3


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


LlamaTokenizer(name_or_path='PY007/TinyLlama-1.1B-Chat-v0.3', vocab_size=32000, model_max_length=1000000000000000019884624838656, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '</s>'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32000: AddedToken("[PAD]", rstrip=True, lstrip=True, single_word=False, normalized=False, special=True),
	32001: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32002: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}


In [31]:
input_string = '我想学习LLM微调'
input_ids = tokenizer(input_string)
print(input_ids['input_ids'])
output_string = tokenizer.decode(input_ids['input_ids'])
print(output_string)

[1, 29871, 30672, 31522, 30415, 231, 188, 163, 2208, 29924, 31935, 31268]
<s> 我想学习LLM微调


# 数据集

In [32]:
# 该数据集已经包含Pretrained、fintune、Reward数据集代码， 仅加载Reward，用于教程
# Pretrained采用加载txt的方式，通用性更好
datasets = load_dataset(datasets_name, 'reward')

pretrain

train_encyclopedia.json: 共36万条，来自医疗百科数据FreedomIntelligence/huatuo_encyclopedia_qa , 拼接 questions 和 answers，形成 text 文本字段，语句通顺，用于预训练注入医疗知识。 medical_book_zh.json: 共8475条，来自医疗教材的文本数据，来源：https://github.com/jind11/MedQA， 原始数据集：google drive ，只对长段落切分为2048字的小段落了。

finetune

train_zh_0.json: 共195万条，来自1）中文医疗对话数据集Toyhom/Chinese-medical-dialogue-data的六个科室医疗问诊数据， 有79万条；2）在线医疗百科 huatuo_encyclopedia_qa ，有36万条；3）医疗知识图谱 huatuo_knowledge_graph_qa，有79万条。三部分合并，共195万条。 train_en_1.json：共11万条，来自英文医疗问诊对话数据Kent0n-Li/ChatDoctor，合并了HealthCareMagic-100k、GenMedGPT-5k 数据集，共11万条。

reward

train.json 共4000条，问题来自中文医疗对话数据集Toyhom/Chinese-medical-dialogue-data的随机4000条提问，response_chosen来自该数据集的医生答复， response_rejected来自本草模型SCIR-HI/Huatuo-Llama-Med-Chinese的答复。

In [33]:
print(datasets)

DatasetDict({
    train: Dataset({
        features: ['question', 'response_chosen', 'response_rejected'],
        num_rows: 3800
    })
    validation: Dataset({
        features: ['question', 'response_chosen', 'response_rejected'],
        num_rows: 100
    })
    test: Dataset({
        features: ['question', 'response_chosen', 'response_rejected'],
        num_rows: 100
    })
})


In [34]:
# 人类的回答为Chosen， 其他LLM的模型的回答作为rejected
sample_index = 5
print("Question: ", datasets['train']['question'][sample_index])
print("response_chosen: ", datasets['train']['response_chosen'][sample_index])
print("response_rejected: ", datasets['train']['response_rejected'][sample_index])

Question:  轻度白内障的临床表现有些什么？
response_chosen:  轻度白内障伴玻璃体混浊
response_rejected:  轻度白内障患者视力下降、眼痛等症状。


# 创建一个Baby-LLaMA(optional)

In [ ]:
# 如果使用Colab或GPU算力显存充足情况， 可忽略当前步骤
# 没有GPU资源的情况，自己创建个baby-llama，参数量极少，但是需要从头开始训练
if debug_mode:
  config = AutoConfig.from_pretrained(model_base_name)
  print(config)
  config.num_attention_heads = 4
  config.num_key_value_heads = 4
  config.num_hidden_layers = 1
  config.hidden_size = 256
  config.intermediate_size = 768
  model = AutoModelForCausalLM.from_config(config)
  print(model)

In [ ]:
# 保存成base mode，从头训练
if debug_mode:
    model.save_pretrained(model_name)
    tokenizer.save_pretrained(model_name)

# Pretrained训练

## 创建Pretrained数据集

In [35]:
def prepare_data_pretrained(example):
    example['question'] = f"{example['question']}{example['response_rejected']}{tokenizer.eos_token}"
    example['question'] = example['question'][:max_length]  #最大长度 128
    example = tokenizer(example['question'])
    return example

datasets_pretrained = datasets.map(prepare_data_pretrained)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [36]:
datasets_pretrained

DatasetDict({
    train: Dataset({
        features: ['question', 'response_chosen', 'response_rejected', 'input_ids', 'attention_mask'],
        num_rows: 3800
    })
    validation: Dataset({
        features: ['question', 'response_chosen', 'response_rejected', 'input_ids', 'attention_mask'],
        num_rows: 100
    })
    test: Dataset({
        features: ['question', 'response_chosen', 'response_rejected', 'input_ids', 'attention_mask'],
        num_rows: 100
    })
})

In [37]:
datasets

DatasetDict({
    train: Dataset({
        features: ['question', 'response_chosen', 'response_rejected'],
        num_rows: 3800
    })
    validation: Dataset({
        features: ['question', 'response_chosen', 'response_rejected'],
        num_rows: 100
    })
    test: Dataset({
        features: ['question', 'response_chosen', 'response_rejected'],
        num_rows: 100
    })
})

In [38]:
# 新增'input_ids', 'token_type_ids', 'attention_mask'
print(datasets['train'])

Dataset({
    features: ['question', 'response_chosen', 'response_rejected'],
    num_rows: 3800
})


## 基于医疗文本创建预训练数据集

In [39]:
from pathlib import Path
from itertools import chain


def tokenize_function(examples):
    output = tokenizer(examples["text"])
    return output

def group_texts(examples):
    concatenated_examples = {k: list(chain(*examples[k])) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

# 如果使用与训练的
if use_pretrained_text_data:
  datasets_pretrained = []
  path = Path(dataset_dir)
  files = [file.name for file in path.glob("*.txt")]
  for idx, file in enumerate(files):
      data_file = os.path.join(path, file)
      filename = ''.join(file.split(".")[:-1])
      cache_path = os.path.join(data_cache_dir, filename)
      os.makedirs(cache_path, exist_ok=True)
      if True:
          cache_dir = os.path.join(data_cache_dir, filename+"_text")
          os.makedirs(cache_dir, exist_ok=True)
          raw_dataset = load_dataset("text", data_files=data_file, cache_dir=cache_dir, keep_in_memory=False)
          print(f"{file} has been loaded")
          tokenized_dataset = raw_dataset.map(
              tokenize_function,
              batched=True,
              num_proc=1,
              remove_columns="text",
              load_from_cache_file=True,
              keep_in_memory=False,
              cache_file_names = {k: os.path.join(cache_dir, 'tokenized.arrow') for k in raw_dataset},
              desc="Running tokenizer on dataset",
          )
          grouped_datasets = tokenized_dataset.map(
              group_texts,
              batched=True,
              num_proc=1,
              load_from_cache_file=True,
              keep_in_memory=False,
              cache_file_names = {k: os.path.join(cache_dir, 'grouped.arrow') for k in tokenized_dataset},
              desc=f"Grouping texts in chunks of {block_size}",
          )
          processed_dataset = grouped_datasets
          processed_dataset.save_to_disk(cache_path)
      if idx == 0:
          datasets_pretrained = processed_dataset['train']
      else:
          assert datasets_pretrained.features.type == processed_dataset["train"].features.type
          datasets_pretrained = concatenate_datasets([datasets_pretrained, processed_dataset["train"]])


  datasets_pretrained = datasets_pretrained.train_test_split(test_size = 0.05)

  print(tokenizer.decode(datasets_pretrained['train'][10]['input_ids']))
  print(tokenizer.decode(datasets_pretrained['test'][10]['input_ids']))

生物化学与分子生物学.txt has been loaded


Saving the dataset (0/1 shards):   0%|          | 0/3110 [00:00<?, ? examples/s]

医学影像学.txt has been loaded


Saving the dataset (0/1 shards):   0%|          | 0/2426 [00:00<?, ? examples/s]

医学微生物学.txt has been loaded


...

传染病学.txt has been loaded


Saving the dataset (0/1 shards):   0%|          | 0/3281 [00:00<?, ? examples/s]

耳鼻咽喉头颈外科学.txt has been loaded


Saving the dataset (0/1 shards):   0%|          | 0/2870 [00:00<?, ? examples/s]

��翼发育、人中长度，嘴唇发育，有无唇裂、腮裂和高腮弓，面部发育是否对称，有无毛发稀疏和颜色异常。注意上部量与下部量比例、指距、手指长度、是否有多指或并指，乳头距离，注意脊柱、胸廓异常，注意关节活动是否异常，注意皮肤和毛发色素、手纹、外生殖器等。注意黄疽、肝脾大、心脏异常听诊音和神经系统症状（肌力、肌张力等），注意一些不正常的汗味或尿味等
障碍，草酰乙酸减少，乙酰CoA进入三狻酸循环受阻，导致乙酰CoA大量堆积，酮体生成增多。<s> (3)丙二酸单酰CoA抑制酮体生成：糖代谢旺盛时，乙酰CoA及拧檬酸增多，别构激活乙酰CoA狻化酶，促进丙二酸单酰CoA合成，后者竞争性抑制肉碱脂酰转移酶I，阻止脂酰CoA进入线粒体进行B氧化，从而�


## 加载base训练模型

In [40]:
del model
if not debug_mode:
    torch.cuda.empty_cache()

In [41]:

# print(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map='auto'
)
model.config.use_cache = False
model.config.pad_token_id = model.config.eos_token_id

if not debug_mode:
    model = get_peft_model(model, lm_lora_config)
    model.print_trainable_parameters()
model.to(device)

config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

trainable params: 3,604,480 || all params: 1,103,665,152 || trainable%: 0.3265918103392287


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32003, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaSdpaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
              )
              (k_proj): lora.Linear4bit(
                (base_layer): Linear4bi

In [42]:
# Question:  轻度白内障的临床表现有些什么？
# test 程序,
prompt = '轻度白内障的临床表现有些什么？'
input_ids = tokenizer(prompt, return_tensors="pt").to(device)
# output = model.generate(**input_ids, max_new_tokens=50, top_k=200, penalty_alpha=1.6, do_sample=True)
output = model.generate(**input_ids, max_new_tokens=100)
response = tokenizer.decode(output[0], skip_special_tokens=True)
print(response)

轻度白内障的临床表现有些什么？

尽管轻度白内障的表现是一种平淡而平淡的内障，但它的临床表现可以很好地反映出其原因。例如，它可能导致睡眠症状、心理症状、心��


## 设置训练参数

In [43]:
# max_steps = 1000
eval_freq = 100
save_freq = 1000
log_freq = 10

training_args = TrainingArguments(
    output_dir=model_pretrained_name,
    dataloader_drop_last=True,
    eval_steps=eval_freq,
    save_steps=save_freq,
    logging_steps=log_freq,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    evaluation_strategy="steps",
    max_steps=max_steps,
    warmup_steps=100,
    gradient_accumulation_steps=1,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    weight_decay=0.05,
    fp16=True

)

## Pretrained模型训练

In [44]:
trainer = Trainer(model=model,
                  args=training_args,
                  train_dataset=datasets_pretrained['train'],
                  eval_dataset=datasets_pretrained['test'],
                  data_collator=data_collator)

/usr/local/lib/python3.10/dist-packages/accelerate/accelerator.py:436: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(


In [45]:
trainer.train()

Step,Training Loss,Validation Loss
100,2.398300,2.400797
200,2.282600,2.282067
300,2.226100,2.241726
400,2.229000,2.212432
500,2.237700,2.193193
600,2.144900,2.180520
700,2.067000,2.169437
800,2.141500,2.163604
900,2.195800,2.160950
1000,2.148100,2.160487


TrainOutput(global_step=1000, training_loss=2.240926630020142, metrics={'train_runtime': 3175.3285, 'train_samples_per_second': 1.26, 'train_steps_per_second': 0.315, 'total_flos': 6378227761152000.0, 'train_loss': 2.240926630020142, 'epoch': 0.05})

In [46]:
# 保存预训练好的模型，这里保存的是adapter
model.save_pretrained(model_pretrained_name)
tokenizer.save_pretrained(model_pretrained_name)

('/content/drive/MyDrive/tinyllama-medical/tinyllama-medical-pretrained/tokenizer_config.json',
 '/content/drive/MyDrive/tinyllama-medical/tinyllama-medical-pretrained/special_tokens_map.json',
 '/content/drive/MyDrive/tinyllama-medical/tinyllama-medical-pretrained/tokenizer.model',
 '/content/drive/MyDrive/tinyllama-medical/tinyllama-medical-pretrained/added_tokens.json')

## 模型合并

https://huggingface.co/docs/peft/conceptual_guides/lora

使用这个函数, merge_and_unload() 具体操作adapter+base model合并当成是基线模型

## Pretrained模型测试

In [47]:
# Question:  轻度白内障的临床表现有些什么？
# test 程序,
prompt = 'Question:轻度白内障的临床表现有些什么？'
input_ids = tokenizer(prompt, return_tensors="pt").to(device)
output = model.generate(**input_ids, max_new_tokens=30,top_k=100,
                        do_sample=True, repetition_penalty=1.6)
response = tokenizer.decode(output[0], skip_special_tokens=True)
print(response)

Question:轻度白内障的临床表现有些什么？Q：主要疾患前发出色因性反应。以小、接近双面一点多重


# SFT训练

In [48]:
del model
# del tensor
# del optimizer
if not debug_mode:
    torch.cuda.empty_cache()

## SFT数据处理

In [49]:
def prepare_sample_text(example):
    text = f"Question: {example['question']}\n\nAnswer: {example['response_rejected']}{tokenizer.eos_token}"
    return text


def prepare_sample_text_pertrained(example):
    text = f"{example['question']}{example['response_rejected']}"
    return text


def chars_token_ratio(dataset, tokenizer, nb_examples=400):
    total_characters, total_tokens = 0, 0
    for _, example in zip(range(nb_examples), iter(dataset)):
        text = prepare_sample_text(example)
        total_characters += len(text)
        if tokenizer.is_fast:
            total_tokens += len(tokenizer(text).tokens())
        else:
            total_tokens += len(tokenizer.tokenize(text))
    return total_characters / total_tokens


def create_sft_datasets(datasets, tokenizer, seq_length=128):

    train_data = datasets["train"]
    valid_data = datasets["test"]

    chars_per_token = chars_token_ratio(train_data, tokenizer)
    print(
        f"The character to token ratio of the dataset is: {chars_per_token:.2f}"
    )

    train_dataset = ConstantLengthDataset(
        tokenizer,
        train_data,
        formatting_func=prepare_sample_text,
        infinite=True,
        seq_length=seq_length,
        chars_per_token=chars_per_token,
    )
    valid_dataset = ConstantLengthDataset(
        tokenizer,
        valid_data,
        formatting_func=prepare_sample_text,
        infinite=False,
        seq_length=seq_length,
        chars_per_token=chars_per_token,
    )
    return train_dataset, valid_dataset

In [50]:
train_data, val_data = create_sft_datasets(datasets, tokenizer)

The character to token ratio of the dataset is: 0.75


## SFT模型加载

In [51]:
model = AutoModelForCausalLM.from_pretrained(
    model_pretrained_name,
    quantization_config=bnb_config if not debug_mode else None ,
    device_map = 'auto'
)
model.config.use_cache = False
model = get_peft_model(model, lm_lora_config)
model.print_trainable_parameters()
model.config.pad_token_id = model.config.eos_token_id

trainable params: 3,604,480 || all params: 1,103,665,152 || trainable%: 0.3265918103392287


In [52]:
# 查看模型参数中的数据类型
for name, param in model.named_parameters():
    print(name, param.dtype)

base_model.model.model.embed_tokens.weight torch.float16
base_model.model.model.layers.0.self_attn.q_proj.base_layer.weight torch.uint8
base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight torch.float32
base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight torch.float32
base_model.model.model.layers.0.self_attn.k_proj.base_layer.weight torch.uint8
base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight torch.float32
base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight torch.float32
base_model.model.model.layers.0.self_attn.v_proj.base_layer.weight torch.uint8
base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight torch.float32
base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight torch.float32
base_model.model.model.layers.0.self_attn.o_proj.base_layer.weight torch.uint8
base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight torch.float32
base_model.model.model.layers.0.

## 模型加载

In [53]:
# train_data.start_iteration = 0
max_steps = 1000
eval_freq = 100
save_freq = 1000
log_freq = 10

training_args = TrainingArguments(
    output_dir=model_sft_name,
    dataloader_drop_last=True,
    evaluation_strategy="steps",
    max_steps=max_steps,
    eval_steps=eval_freq,
    save_steps=save_freq,
    logging_steps=log_freq,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_steps=100,
    gradient_accumulation_steps=1,
    weight_decay=0.05,
    #     ddp_find_unused_parameters=False,
    fp16=True,
)

In [54]:
trainer = Trainer(model=model,
                  args=training_args,
                  train_dataset=train_data,
                  eval_dataset=val_data,
                  data_collator=data_collator)

/usr/local/lib/python3.10/dist-packages/accelerate/accelerator.py:436: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(


In [55]:
trainer.train()

Step,Training Loss,Validation Loss
100,2.360900,2.280642
200,2.143500,2.221813
300,2.276300,2.297896
400,2.241500,2.297820
500,2.296300,2.297521
600,2.397900,2.297486
700,2.243400,2.297282
800,2.267300,2.297106
900,2.233200,2.297016
1000,2.308300,2.296911


/usr/local/lib/python3.10/dist-packages/trl/trainer/utils.py:462: UserWarning: The dataset reached end and the iterator is reset to the start.
  warnings.warn("The dataset reached end and the iterator is reset to the start.")
/usr/local/lib/python3.10/dist-packages/trl/trainer/utils.py:462: UserWarning: The dataset reached end and the iterator is reset to the start.
  warnings.warn("The dataset reached end and the iterator is reset to the start.")


TrainOutput(global_step=1000, training_loss=2.278060272216797, metrics={'train_runtime': 620.7195, 'train_samples_per_second': 12.888, 'train_steps_per_second': 1.611, 'total_flos': 6378227761152000.0, 'train_loss': 2.278060272216797, 'epoch': 2.11})

In [56]:
# 保存预训练好的模型
model.save_pretrained(model_sft_name)
tokenizer.save_pretrained(model_sft_name)

('/content/drive/MyDrive/tinyllama-medical/tinyllama-medical-SFT/tokenizer_config.json',
 '/content/drive/MyDrive/tinyllama-medical/tinyllama-medical-SFT/special_tokens_map.json',
 '/content/drive/MyDrive/tinyllama-medical/tinyllama-medical-SFT/tokenizer.model',
 '/content/drive/MyDrive/tinyllama-medical/tinyllama-medical-SFT/added_tokens.json')

## SFT 模型测试

In [57]:
# Question:  轻度白内障的临床表现有些什么？

prompt = '轻度白内障的临床表现有些什么?'
input_ids = tokenizer(prompt, return_tensors="pt").to(device)
output = model.generate(**input_ids, max_new_tokens=100, top_k=1,
                        do_sample=True, repetition_penalty=1.2)
response = tokenizer.decode(output[0], skip_special_tokens=True)
print(response)

轻度白内障的临床表现有些什么?患者可以不会症状出来，还是要去治好了就能解除一下。我想问到如果没有手足而无法用手力进行自助体操时应该多少次每天才能完全消化水平和保持正常情形？请教。Answer: 根据国家统计局所知道人口年代之后生长性


## 上传模型到Huggingface hub

In [ ]:
# 登陆Huggingface， 这里的Acesse Token需要Write权限
from huggingface_hub import notebook_login
from huggingface_hub import create_repo
notebook_login()

In [ ]:
# 创建仓库
create_repo("xxx_TinyLLaMA_medical_sft")

In [ ]:
# 上传Model和Tokenizer
model.push_to_hub("xxx_TinyLLaMA_medical_sft")
tokenizer.push_to_hub("xxx_TinyLLaMA_medical_sft")

# DPO

## DPO创建RM数据

In [ ]:
def rm_data(split: str) -> Dataset:

    print(split)
    dataset = load_dataset("shibing624/medical", 'reward', split=split)

    def split_prompt_and_responses(sample) -> Dict[str, str]:
        return {
            "prompt": sample["question"],
            "chosen": sample["response_chosen"],
            "rejected": sample["response_rejected"],
        }

    return dataset.map(split_prompt_and_responses)


train_data = rm_data('train')
val_data = rm_data('test')

tokenizer.eos_token = tokenizer.pad_token
print(tokenizer.pad_token)
print(tokenizer.eos_token)

In [59]:
print(train_data['prompt'][1])

两只脚明显大小不一样，腿也不一样粗，该怎么办，两只脚明显大小不一样，腿也不一样粗，该怎么办，需要做什么检查


## 加载SFT模型

In [60]:
dpo_model = AutoModelForCausalLM.from_pretrained(model_sft_name)
dpo_model_ref = AutoModelForCausalLM.from_pretrained(model_sft_name)

## DPO训练参数

In [ ]:
dpo_beta = 0.1
dpo_max_length = 128
dpo_max_target_length = 128
dpo_max_prompt_length = 128
dpo_learning_rate = 0.1

# 4. initialize training arguments:
dpo_training_args = TrainingArguments(
    per_device_train_batch_size=batch_size,
    max_steps=max_steps,
    remove_unused_columns=False,
    gradient_accumulation_steps=1,
    learning_rate=dpo_learning_rate,
    evaluation_strategy="steps",
    logging_first_step=True,
    eval_steps=100,
    output_dir=model_dpo_name,
)

dpo_trainer = DPOTrainer(
    dpo_model,
    dpo_model_ref,
    args=dpo_training_args,
    beta=dpo_beta,
    train_dataset=train_data,
    eval_dataset=val_data,
    tokenizer=tokenizer,
    max_length=dpo_max_length,
)

/usr/local/lib/python3.10/dist-packages/trl/trainer/dpo_trainer.py:307: UserWarning: `max_prompt_length` is not set in the DPOTrainer's init it will default to `128` by default, but you should do it yourself in the future.
  warnings.warn(


In [ ]:
dpo_trainer.train()

In [ ]:
dpo_model.save_pretrained(model_dpo_name)
tokenizer.save_pretrained(model_dpo_name)

## DPO模型测试

In [ ]:
# Question:  轻度白内障的临床表现有些什么？
# test 程序,

# 轻度白内障的临床表现有些什么？轻度白内障伴玻璃体混浊
# 轻度白内障的临床表现有些什么？轻度白内障患者视力下降、眼痛等症状。

prompt = '轻度白内障的临床表现'
input_ids = tokenizer(prompt, return_tensors="pt").to(device)
output = dpo_model.generate(**input_ids, max_new_tokens=20)
response = tokenizer.decode(output[0], skip_special_tokens=True)
print(response)